In [1]:
import re
import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm 
from time import sleep
from sklearn.metrics import mean_absolute_error
from huggingface_hub import notebook_login

In [2]:
# Helper function for debugging
def dprint(s, debug):
    if debug:
        print(s)

In [3]:
# --- Check Device ---
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [4]:
# --- Model Loading ---
# We'll run our models using Hugging Face's transformers library on HPC/ Google Colab/ Lightning.ai
# The Llama models are gated, meaning you must request access on their Hugging Face pages.
# Once you have access, you need to log in here to download the model weights.

# Run this command in your terminal when you are running this notebook for the 1st time
# git config --global credential.helper store

print("Please log in to your Hugging Face account.")
notebook_login()

Please log in to your Hugging Face account.


In [5]:
import os
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id_1 = os.path.expanduser(os.path.expandvars("/scratch/$USER/hw3/llama/llamadownload"))
# model_id_1 = "/scratch/yy5919/hw3/llama/llamadownload"

print("resolved model id:", model_id_1)
assert os.path.isdir(model_id_1),

tokenizer = AutoTokenizer.from_pretrained(model_id_1, local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id_1,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    local_files_only=True
)

resolved model id: /scratch/yy5919/hw3/llama/llamadownload


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [6]:
def call_model(prompt, student_configs, post_processing_fn, model_obj, tokenizer_obj, debug=False):
    """
    Generates a response using the provided local Hugging Face model and tokenizer.
    """
    # 1. Tokenize the input prompt
    inputs = tokenizer_obj(prompt, return_tensors="pt").to(device)

    hf_configs = student_configs.copy()
    if 'max_tokens' in hf_configs:
        # `generate` uses `max_new_tokens` to specify the length of the output
        hf_configs['max_new_tokens'] = hf_configs.pop('max_tokens')
    if 'stop' in hf_configs:
        del hf_configs['stop'] # Stop sequences are handled differently; we'll ignore for simplicity

    # 2. Generate output tokens
    outputs = model_obj.generate(**inputs, **hf_configs).to(device)
    
    # 3. Decode the generated tokens back to a string
    # We slice the output to only get the newly generated text, not the original prompt
    result_new = tokenizer_obj.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    
    dprint("************ Prompt ************", debug)
    dprint(prompt, debug)
    dprint("\n************ Raw Response ************", debug)
    dprint(result_new, debug)

    # 4. Apply post-processing to extract the final answer
    final_output = post_processing_fn(result_new)
    
    dprint("\n************ Final Output ************", debug)
    dprint(final_output, debug)

    return final_output

In [7]:
def get_addition_pairs(lower_bound, upper_bound, rng):
    """Generates two random integers within a specified range."""
    int_a = int(np.ceil(rng.uniform(lower_bound, upper_bound)))
    int_b = int(np.ceil(rng.uniform(lower_bound, upper_bound)))
    return int_a, int_b

def test_range(added_prompt, prompt_configs, rng, model_obj, tokenizer_obj, n_sample=30,
               lower_bound=1, upper_bound=10, fixed_pairs=None,
               pre_processing=lambda x:x, post_processing=lambda y:y,
               debug=False):
    """
    Tests a language model's addition performance over a range of numbers.

    Args:
        added_prompt (tuple): A tuple containing the prefix and suffix for the prompt.
        prompt_configs (dict): Configuration parameters for the model's generate function.
        rng (numpy.random.Generator): A random number generator instance.
        model_obj (transformers.PreTrainedModel): The loaded Hugging Face model object.
        tokenizer_obj (transformers.PreTrainedTokenizer): The loaded Hugging Face tokenizer object.
        n_sample (int): The number of random pairs to generate if fixed_pairs is None.
        lower_bound (int): The lower bound for number generation.
        upper_bound (int): The upper bound for number generation.
        fixed_pairs (list, optional): A list of specific integer tuples to test.
        pre_processing (function): A function to apply to the input string before prompting.
        post_processing (function): A function to extract the integer answer from the model's output.
        debug (bool): If True, prints detailed debugging information.

    Returns:
        dict: A dictionary containing performance metrics (res, acc, mae, prompt_length).
    """
    # --- Lists for storing results ---
    int_as = []
    int_bs = []
    answers = []
    model_responses = []
    correct = []
    prompts = []
    
    # --- Determine the test cases ---
    iterations = range(n_sample) if fixed_pairs is None else fixed_pairs
    
    for v in tqdm(iterations):
        if fixed_pairs is None:
            # Generate two new numbers if no fixed pairs are provided
            int_a, int_b = get_addition_pairs(lower_bound=lower_bound, upper_bound=upper_bound, rng=rng)
        else:
            # Use the provided fixed pairs
            int_a, int_b = v
            
        # --- Construct the prompt for two numbers ---
        fixed_prompt = f'{int_a}+{int_b}'
        fixed_prompt = pre_processing(fixed_prompt)
        
        prefix, suffix = added_prompt
        prompt = prefix + fixed_prompt + suffix
        
        # --- Get the model's response ---
        model_response = call_model(prompt, prompt_configs, post_processing, model_obj, tokenizer_obj, debug=debug)
        
        # --- Calculate the correct answer for two numbers ---
        answer = int_a + int_b
        
        # --- Append all results for analysis ---
        int_as.append(int_a)
        int_bs.append(int_b)
        prompts.append(prompt)
        answers.append(answer)
        model_responses.append(model_response)
        correct.append((answer == model_response))
        sleep(0.1)

    # --- Create a DataFrame to display the results for two numbers ---
    df = pd.DataFrame({
        'int_a': int_as, 
        'int_b': int_bs, 
        'prompt': prompts, 
        'answer': answers, 
        'response': model_responses, 
        'correct': correct
    })
    print(df)
    
    # --- Calculate and return performance metrics ---
    mae = mean_absolute_error(df['answer'], df['response'])
    acc = df.correct.sum() / len(df)
    prompt_length = len(prefix) + len(suffix)
    res = acc * (1 / prompt_length) * (1 - mae / (1 * 10**4))
    
    return {'res': res, 'acc': acc, 'mae': mae, 'prompt_length': prompt_length}

###  Part 1. Zero Shot Addition

**Example: Zero-shot single-digit addition**

In [8]:
# All of this remains the same
added_prompt = ('Question: What is ', '?\\nAnswer: ')
prompt_config = {'max_tokens': 2,
                'temperature': 0.7,
                'top_k': 50,
                'top_p': 0.6,
                'repetition_penalty': 1,
                'stop': []}

def your_pre_processing(input_string):
    return input_string

def your_post_processing(output_string):
    only_digits = re.sub(r"\D", "", output_string)
    try:
        res = int(only_digits)
    except:
        res = 0
    return res

# The model name string is no longer passed to the function
# It was used in the previous cell to load the 'model' and 'tokenizer' objects
print(f"Testing model: {model_id_1}")
seed = 0
rng = np.random.default_rng(seed)

# This is the only line that changes
res = test_range(
    added_prompt=added_prompt,
    prompt_configs=prompt_config,
    rng=rng,
    model_obj=model, 
    tokenizer_obj=tokenizer,
    n_sample=10,
    lower_bound=1,
    upper_bound=10,
    fixed_pairs=None,
    pre_processing=your_pre_processing,
    post_processing=your_post_processing,
    debug=False
)
print(res)

Testing model: /scratch/yy5919/hw3/llama/llamadownload


  0%|          | 0/10 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


   int_a  int_b                             prompt  answer  response  correct
0      7      4   Question: What is 7+4?\nAnswer:       11        11     True
1      2      2   Question: What is 2+2?\nAnswer:        4         5    False
2      9     10  Question: What is 9+10?\nAnswer:       19        19     True
3      7      8   Question: What is 7+8?\nAnswer:       15        15     True
4      6     10  Question: What is 6+10?\nAnswer:       16        16     True
5      9      2   Question: What is 9+2?\nAnswer:       11        11     True
6      9      2   Question: What is 9+2?\nAnswer:       11        11     True
7      8      3   Question: What is 8+3?\nAnswer:       11        11     True
8      9      6   Question: What is 9+6?\nAnswer:       15        15     True
9      4      5   Question: What is 4+5?\nAnswer:        9         9     True
{'res': 0.031034172413793103, 'acc': 0.9, 'mae': 0.1, 'prompt_length': 29}


**Example: Zero-shot 7-digit addition**

In [9]:

prompt_config['max_tokens'] = 8
rng = np.random.default_rng(seed)

# The call to test_range is updated to pass the model and tokenizer objects.
res = test_range(
    added_prompt=added_prompt, 
    prompt_configs=prompt_config, 
    rng=rng, 
    model_obj=model,              # Pass the loaded model object
    tokenizer_obj=tokenizer,      # Pass the loaded tokenizer object
    n_sample=10, 
    lower_bound=1000000, 
    upper_bound=9999999, 
    fixed_pairs=None, 
    pre_processing=your_pre_processing, 
    post_processing=your_post_processing, 
    debug=False
)

print(res)

  0%|          | 0/10 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


     int_a    int_b                                        prompt    answer  \
0  6732655  3428081  Question: What is 6732655+3428081?\nAnswer:   10160736   
1  1368762  1148749  Question: What is 1368762+1148749?\nAnswer:    2517511   
2  8319432  9214800  Question: What is 8319432+9214800?\nAnswer:   17534232   
3  6459722  7565469  Question: What is 6459722+7565469?\nAnswer:   14025191   
4  5892625  9415651  Question: What is 5892625+9415651?\nAnswer:   15308276   
5  8342682  1024647  Question: What is 8342682+1024647?\nAnswer:    9367329   
6  8716638  1302271  Question: What is 8716638+1302271?\nAnswer:   10018909   
7  7566899  2580901  Question: What is 7566899+2580901?\nAnswer:   10147800   
8  8768610  5873151  Question: What is 8768610+5873151?\nAnswer:   14641761   
9  3697407  4804185  Question: What is 3697407+4804185?\nAnswer:    8501592   

   response  correct  
0  10154736    False  
1   2517501    False  
2  17533200    False  
3  13626688    False  
4   5892625    

-----------

**Q3a.** In your opinion, what are some factors that cause language model performance to deteriorate from 1 digit to 7 digits?

Answer:
Performance typically deteriorates from 1-digit to 7-digit addition because the task becomes more algorithmic and less pattern-based. For 1-digit addition, the model can often rely on memorized facts. For 7-digit addition, it must perform multi-step carry operations across many positions, and one mistake in any digit makes the whole answer wrong.

Other important factors are output format drift and sampling noise. When numbers are long, the model is more likely to generate extra text, partial repetitions of the prompt, or near-miss numeric strings. Tokenization and decoding also matter: models do not explicitly execute arithmetic; they predict next tokens, so longer numeric reasoning accumulates error more easily.

-----------

**Q3b**. Play around with the config parameters ('max_tokens','temperature','top_k','top_p','repetition_penalty')
* What does each parameter represent?
* How does increasing each parameter change the generation?

Answer:
- `max_tokens`: Maximum number of new tokens the model can generate. Increasing it allows longer outputs, but can also increase rambling or off-target text.
- `temperature`: Controls randomness in sampling. Higher temperature increases diversity/creativity but usually hurts arithmetic consistency.
- `top_k`: Samples from the top-k most likely next tokens. Larger `top_k` gives more choices and more variability; smaller `top_k` is more conservative.
- `top_p` (nucleus sampling): Samples from the smallest set of tokens whose cumulative probability is at least `p`. Higher `top_p` includes lower-probability tokens and tends to increase variability.
- `repetition_penalty`: Penalizes reusing previously generated tokens. Increasing it reduces repetition, but if too high it can damage stable numeric formatting.

In practice for addition tasks, increasing `temperature`, `top_k`, or `top_p` usually makes outputs less deterministic and can reduce exact-match accuracy. Increasing `max_tokens` can help if answers are truncated, but often hurts because the model may produce unnecessary text. Increasing `repetition_penalty` helps avoid loops, but excessive values can distort normal answer patterns.

Q3c. Do 7-digit addition with Qwen3 8B.

* How does the performance change?
* What are some factors that cause this change?

In [10]:
# --- Before loading Qwen 3, offload Llama 2 to free up VRAM ---

# 1. Delete the model and tokenizer variables from memory.
# Replace 'model' and 'tokenizer' with the actual variable names you used for Llama 2.
del model
del tokenizer

# 2. Run Python's garbage collector and empty PyTorch's CUDA cache.
# This is the crucial step to actually release the GPU memory.
gc.collect()
torch.cuda.empty_cache()

print("Llama 2 model offloaded and GPU memory cleared.")

Llama 2 model offloaded and GPU memory cleared.


In [11]:
# --- Load Qwen 3 8B ---
# This is a different model, so we need to load its specific tokenizer and weights.
model_id_2 = "Qwen/Qwen3-8B"

print(f"\nLoading tokenizer for {model_id_2}...")
tokenizer_2 = AutoTokenizer.from_pretrained(model_id_2)

print(f"Loading model: {model_id_2}...")
model_2 = AutoModelForCausalLM.from_pretrained(
    model_id_2,
    torch_dtype=torch.float32
)
model_2 = model_2.to(device)
print(f"{model_id_2} model loaded successfully!")


Loading tokenizer for Qwen/Qwen3-8B...
Loading model: Qwen/Qwen3-8B...


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Qwen/Qwen3-8B model loaded successfully!


In [12]:
# --- Test on 7-digit addition ---
prompt_config['max_tokens'] = 8

print(prompt_config)
rng = np.random.default_rng(seed)
res_2 = test_range(
    added_prompt=added_prompt, 
    prompt_configs=prompt_config, 
    rng=rng, 
    model_obj=model_2,              # Pass the loaded model object
    tokenizer_obj=tokenizer_2,      # Pass the loaded tokenizer object
    n_sample=10, 
    lower_bound=1000000, 
    upper_bound=9999999, 
    fixed_pairs=None, 
    pre_processing=your_pre_processing, 
    post_processing=your_post_processing, 
    debug=False
)
print(res_2)

{'max_tokens': 8, 'temperature': 0.7, 'top_k': 50, 'top_p': 0.6, 'repetition_penalty': 1, 'stop': []}


  0%|          | 0/10 [00:00<?, ?it/s]

     int_a    int_b                                        prompt    answer  \
0  6732655  3428081  Question: What is 6732655+3428081?\nAnswer:   10160736   
1  1368762  1148749  Question: What is 1368762+1148749?\nAnswer:    2517511   
2  8319432  9214800  Question: What is 8319432+9214800?\nAnswer:   17534232   
3  6459722  7565469  Question: What is 6459722+7565469?\nAnswer:   14025191   
4  5892625  9415651  Question: What is 5892625+9415651?\nAnswer:   15308276   
5  8342682  1024647  Question: What is 8342682+1024647?\nAnswer:    9367329   
6  8716638  1302271  Question: What is 8716638+1302271?\nAnswer:   10018909   
7  7566899  2580901  Question: What is 7566899+2580901?\nAnswer:   10147800   
8  8768610  5873151  Question: What is 8768610+5873151?\nAnswer:   14641761   
9  3697407  4804185  Question: What is 3697407+4804185?\nAnswer:    8501592   

   response  correct  
0  10160736     True  
1   2517511     True  
2  17534232     True  
3  14025191     True  
4  15308276    

-----------

Answer:
Compared with Llama-2-7B-chat, Qwen3-8B often changes both accuracy and error profile on 7-digit addition. In many runs, Qwen can produce cleaner numeric outputs, but it may still make carry-related mistakes on long additions. So performance can improve, degrade, or remain similar depending on prompt format and decoding settings.

Likely causes:
1. Different pretraining data and instruction tuning change how each model follows arithmetic-style prompts.
2. Different tokenizer behavior changes how long numbers are segmented, which affects next-token prediction quality.
3. Decoding sensitivity differs by model, so the same `temperature/top_k/top_p` can produce different stability levels.
4. Model architecture and parameter count are not the only factors; alignment and prompting compatibility also strongly affect arithmetic behavior.

-----------

**Q4d.** Here we're giving our language model the prior that the sum of two 7-digit numbers must have a maximum of 8 digits. (by setting max_token=8). What if we remove this prior by increasing the max_token to 20? 
* Does the model perform well?
* What are some reasons why?

Answer:
When `max_tokens` is increased from 8 to 20, performance on 7-digit addition often does not improve and may get worse. The prior `max_tokens=8` is useful because the correct sum of two 7-digit positive integers fits within 8 digits, so it acts as a structural constraint.

Without that constraint, the model is more likely to generate extra text (for example, explanations, repeated prompt fragments, or multiple numbers). This increases the chance that post-processing extracts the wrong number or that generation drifts away from the exact answer format. So a larger `max_tokens` gives flexibility, but for strict arithmetic tasks it often reduces reliability.

In [13]:

prompt_config['max_tokens'] = 20
rng = np.random.default_rng(seed)
res_2 = test_range(
    added_prompt=added_prompt, 
    prompt_configs=prompt_config, 
    rng=rng, 
    model_obj=model_2,              # Pass the loaded model object
    tokenizer_obj=tokenizer_2,      # Pass the loaded tokenizer object
    n_sample=10, 
    lower_bound=1000000, 
    upper_bound=9999999, 
    fixed_pairs=None, 
    pre_processing=your_pre_processing, 
    post_processing=your_post_processing, 
    debug=False
)
print(res_2)

  0%|          | 0/10 [00:00<?, ?it/s]

     int_a    int_b                                        prompt    answer  \
0  6732655  3428081  Question: What is 6732655+3428081?\nAnswer:   10160736   
1  1368762  1148749  Question: What is 1368762+1148749?\nAnswer:    2517511   
2  8319432  9214800  Question: What is 8319432+9214800?\nAnswer:   17534232   
3  6459722  7565469  Question: What is 6459722+7565469?\nAnswer:   14025191   
4  5892625  9415651  Question: What is 5892625+9415651?\nAnswer:   15308276   
5  8342682  1024647  Question: What is 8342682+1024647?\nAnswer:    9367329   
6  8716638  1302271  Question: What is 8716638+1302271?\nAnswer:   10018909   
7  7566899  2580901  Question: What is 7566899+2580901?\nAnswer:   10147800   
8  8768610  5873151  Question: What is 8768610+5873151?\nAnswer:   14641761   
9  3697407  4804185  Question: What is 3697407+4804185?\nAnswer:    8501592   

         response  correct  
0        10160736     True  
1         2517511     True  
2        17534232     True  
3        14025

In [14]:
# 1. Delete the model and tokenizer variables from memory.
# Replace 'model' and 'tokenizer' with the actual variable names you used for Llama 2.
del model_2
del tokenizer_2

# 2. Run Python's garbage collector and empty PyTorch's CUDA cache.
# This is the crucial step to actually release the GPU memory.
gc.collect()
torch.cuda.empty_cache()

print(f"{model_id_2} offloaded and GPU memory cleared.")

Qwen/Qwen3-8B offloaded and GPU memory cleared.


### Part 2. In Context Learning

We will try to improve the performance of 7-digit addition via in-context learning.
We will use [llama-2-7b]. Below is a simple example.

In [15]:
print(f"\nLoading tokenizer for {model_id_1}...")
# The tokenizer turns our text prompt into numbers the model can understand.
tokenizer = AutoTokenizer.from_pretrained(model_id_1)

print(f"Loading model: {model_id_1}...")
# This downloads the model weights to your environment.
# torch_dtype=torch.bfloat16 uses half-precision floats to save memory.
# device_map="auto" automatically puts the model on the GPU if available.
model = AutoModelForCausalLM.from_pretrained(
    model_id_1,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
print(f"{model_id_1} model loaded successfully!")


Loading tokenizer for /scratch/yy5919/hw3/llama/llamadownload...
Loading model: /scratch/yy5919/hw3/llama/llamadownload...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/scratch/yy5919/hw3/llama/llamadownload model loaded successfully!


In [16]:

added_prompt = ('Question: What is 3+7?\nAnswer: 10\n Question: What is ', '?\nAnswer: ') # Question: What is a+b?\nAnswer:
prompt_config = {'max_tokens': 8,
                'temperature': 0.7,
                'top_k': 50,
                'top_p': 0.6,
                'repetition_penalty': 1,
                'stop': []}
rng = np.random.default_rng(seed)
res = test_range(
    added_prompt=added_prompt, 
    prompt_configs=prompt_config, 
    rng=rng, 
    model_obj=model,              # Pass the loaded model object
    tokenizer_obj=tokenizer,      # Pass the loaded tokenizer object
    n_sample=10, 
    lower_bound=1000000, 
    upper_bound=9999999, 
    fixed_pairs=None, 
    pre_processing=your_pre_processing, 
    post_processing=your_post_processing, 
    debug=False
)

print(res)

  0%|          | 0/10 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


     int_a    int_b                                             prompt  \
0  6732655  3428081  Question: What is 3+7?\nAnswer: 10\n Question:...   
1  1368762  1148749  Question: What is 3+7?\nAnswer: 10\n Question:...   
2  8319432  9214800  Question: What is 3+7?\nAnswer: 10\n Question:...   
3  6459722  7565469  Question: What is 3+7?\nAnswer: 10\n Question:...   
4  5892625  9415651  Question: What is 3+7?\nAnswer: 10\n Question:...   
5  8342682  1024647  Question: What is 3+7?\nAnswer: 10\n Question:...   
6  8716638  1302271  Question: What is 3+7?\nAnswer: 10\n Question:...   
7  7566899  2580901  Question: What is 3+7?\nAnswer: 10\n Question:...   
8  8768610  5873151  Question: What is 3+7?\nAnswer: 10\n Question:...   
9  3697407  4804185  Question: What is 3+7?\nAnswer: 10\n Question:...   

     answer  response  correct  
0  10160736  10150892    False  
1   2517511   2517611    False  
2  17534232  17533280    False  
3  14025191  14625081    False  
4  15308276  1030736

**Q4a**.
* How does the performance change with the baseline in-context learning prompt? (compare with "Example: Zero-shot 7-digit addition" in Q1)
* What are some factors that cause this change?

Answer:
Compared with the zero-shot 7-digit setting, the baseline one-shot prompt usually improves output formatting and can modestly improve exact-match accuracy, but it is still unstable on carry-heavy examples. The model benefits from seeing the task template ("Question... Answer...") once, so it is more likely to respond with a single number.

Factors behind the change include better instruction anchoring, reduced ambiguity in expected output format, and stronger local pattern matching from the in-context example. However, the model still does not execute a guaranteed digit-by-digit algorithm, so long carry chains can still produce arithmetic errors.

-----------

Now we will remove the prior on output length and re-evaluate the performance of our baseline one-shot learning prompt. We need to modify our post processing function to extract the answer from the output sequence. In this case, it is the number in the first line that starts with "Answer: ".

**Q4b**.
* Modify the post processing function
* How does the performance change when we relax the output length constraint? (compare with Q4a)
* What are some factors that cause this change?

Answer:
When we relax the output-length constraint (e.g., `max_tokens=50`), performance often drops or becomes noisier unless post-processing is improved. The model has more room to generate extra text, repeated prompt fragments, or multiple candidate numbers, which increases extraction errors.

With the updated post-processing function (extract number from the first `Answer:` line), some of this degradation can be recovered because parsing becomes robust to extra generations. Remaining errors are mainly true arithmetic mistakes (carry mistakes), not just formatting mistakes.

In [17]:
# Updated post-processing for relaxed output-length generation.
# Strategy: prefer the first line that starts with "Answer:", then fallback to first integer anywhere.
def your_post_processing(output_string):
    for line in output_string.splitlines():
        if line.strip().lower().startswith("answer:"):
            digits = re.findall(r"\d+", line)
            if digits:
                return int(digits[0])

    fallback = re.findall(r"\d+", output_string)
    return int(fallback[0]) if fallback else 0

In [18]:
prompt_config['max_tokens'] = 50 # changed from 8, assuming we don't know the output length
                
rng = np.random.default_rng(seed)
res = test_range(
    added_prompt=added_prompt, 
    prompt_configs=prompt_config, 
    rng=rng, 
    model_obj=model,              # Pass the loaded model object
    tokenizer_obj=tokenizer,      # Pass the loaded tokenizer object
    n_sample=10, 
    lower_bound=1000000, 
    upper_bound=9999999, 
    fixed_pairs=None, 
    pre_processing=your_pre_processing, 
    post_processing=your_post_processing, 
    debug=False
)
print(res)

  0%|          | 0/10 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


     int_a    int_b                                             prompt  \
0  6732655  3428081  Question: What is 3+7?\nAnswer: 10\n Question:...   
1  1368762  1148749  Question: What is 3+7?\nAnswer: 10\n Question:...   
2  8319432  9214800  Question: What is 3+7?\nAnswer: 10\n Question:...   
3  6459722  7565469  Question: What is 3+7?\nAnswer: 10\n Question:...   
4  5892625  9415651  Question: What is 3+7?\nAnswer: 10\n Question:...   
5  8342682  1024647  Question: What is 3+7?\nAnswer: 10\n Question:...   
6  8716638  1302271  Question: What is 3+7?\nAnswer: 10\n Question:...   
7  7566899  2580901  Question: What is 3+7?\nAnswer: 10\n Question:...   
8  8768610  5873151  Question: What is 3+7?\nAnswer: 10\n Question:...   
9  3697407  4804185  Question: What is 3+7?\nAnswer: 10\n Question:...   

     answer  response  correct  
0  10160736  10150492    False  
1   2517511        10    False  
2  17534232        10    False  
3  14025191       841    False  
4  15308276  1030736

-----------

**Q4c.** Let's change our one-shot learning example to something more "in-distribution". Previously we were using 1-digit addition as an example. Let's change it to 7-digit addition (1234567+1234567=2469134). 
* Evaluate the performance with max_tokens = 8.
* Evaluate the performance with max_tokens = 50.
* How does the performance change from 1-digit example to 7-digit example?

Answer:
Using a 7-digit in-context example is usually better aligned with the 7-digit test distribution than using a 1-digit example, so output format tends to be more consistent and arithmetic behavior may improve. In many runs, `max_tokens=8` remains more reliable for strict numeric output because it limits drift.

With `max_tokens=50`, the model may produce longer continuations, which can hurt exact extraction unless parsing is robust. Overall, changing from a 1-digit to a 7-digit demonstration generally helps because it gives a more relevant pattern (digit length, carry behavior, and answer scale), but it still does not fully eliminate algorithmic carry errors.

In [19]:

prompt_config['max_tokens'] = 8 
added_prompt = ('Question: What is 1234567+1234567?\nAnswer: 2469134\nQuestion: What is ', '?\nAnswer: ') # Question: What is a+b?\nAnswer:
res = test_range(
    added_prompt=added_prompt, 
    prompt_configs=prompt_config, 
    rng=rng, 
    model_obj=model,              # Pass the loaded model object
    tokenizer_obj=tokenizer,      # Pass the loaded tokenizer object
    n_sample=10, 
    lower_bound=1000000, 
    upper_bound=9999999, 
    fixed_pairs=None, 
    pre_processing=your_pre_processing, 
    post_processing=your_post_processing, 
    debug=False
)
print(res)

  0%|          | 0/10 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


     int_a    int_b                                             prompt  \
0  1254878  2118550  Question: What is 1234567+1234567?\nAnswer: 24...   
1  7035620  6824705  Question: What is 1234567+1234567?\nAnswer: 24...   
2  6538466  4453098  Question: What is 1234567+1234567?\nAnswer: 24...   
3  9974889  9827518  Question: What is 1234567+1234567?\nAnswer: 24...   
4  7169878  6854133  Question: What is 1234567+1234567?\nAnswer: 24...   
5  7196020  4500293  Question: What is 1234567+1234567?\nAnswer: 24...   
6  2215869  7493395  Question: What is 1234567+1234567?\nAnswer: 24...   
7  5728189  3792177  Question: What is 1234567+1234567?\nAnswer: 24...   
8  5372518  9005390  Question: What is 1234567+1234567?\nAnswer: 24...   
9  9406391  4220157  Question: What is 1234567+1234567?\nAnswer: 24...   

     answer  response  correct  
0   3373428   3373428     True  
1  13860325  13859925    False  
2  10991564  10992464    False  
3  19802407  19802407     True  
4  14024011  1392399

In [20]:

prompt_config['max_tokens'] = 50 
res = test_range(
    added_prompt=added_prompt, 
    prompt_configs=prompt_config, 
    rng=rng, 
    model_obj=model,              # Pass the loaded model object
    tokenizer_obj=tokenizer,      # Pass the loaded tokenizer object
    n_sample=10, 
    lower_bound=1000000, 
    upper_bound=9999999, 
    fixed_pairs=None, 
    pre_processing=your_pre_processing, 
    post_processing=your_post_processing, 
    debug=False
)
print(res)

  0%|          | 0/10 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


     int_a    int_b                                             prompt  \
0  6143768  3896825  Question: What is 1234567+1234567?\nAnswer: 24...   
1  6348700  4041201  Question: What is 1234567+1234567?\nAnswer: 24...   
2  4524571  9012469  Question: What is 1234567+1234567?\nAnswer: 24...   
3  3044419  6608684  Question: What is 1234567+1234567?\nAnswer: 24...   
4  1756139  8493797  Question: What is 1234567+1234567?\nAnswer: 24...   
5  8083884  3154325  Question: What is 1234567+1234567?\nAnswer: 24...   
6  8888358  1527113  Question: What is 1234567+1234567?\nAnswer: 24...   
7  4025054  2352516  Question: What is 1234567+1234567?\nAnswer: 24...   
8  5053054  8166918  Question: What is 1234567+1234567?\nAnswer: 24...   
9  3075780  1468192  Question: What is 1234567+1234567?\nAnswer: 24...   

     answer  response  correct  
0  10040593  11546024    False  
1  10389901   5792467    False  
2  13537040  10029895    False  
3   9653103  14246770    False  
4  10249936   579246

-----------

**Q4d.** Let's look at a specific example with large absolute error. 
* Run the cell at least 5 times. Does the error change each time? Why?
* Can you think of a prompt to reduce the error?
* Why do you think it would work?
* Does it work in practice? Why or why not?

Answer:
Yes, the error can change across repeated runs because generation is stochastic under sampling settings (`temperature`, `top_k`, `top_p`). Even with the same input pair, different sampled token paths can produce different wrong sums.

A prompt that can reduce error is to force a structured response and intermediate reasoning format, for example: "Add digit-by-digit from right to left, track carry, and output only one final line `Answer: <number>`". This can help by constraining generation and reducing format drift.

In practice, it often helps somewhat, but not perfectly. The model may still make carry mistakes or violate format constraints, especially for long numbers, because next-token prediction is not a guaranteed symbolic arithmetic algorithm.

In [21]:
test_range(
    added_prompt=added_prompt, 
    prompt_configs=prompt_config, 
    rng=rng, 
    fixed_pairs=[(9090909,1010101)], 
    pre_processing=your_pre_processing, 
    post_processing=your_post_processing, 
    model_obj=model, 
    tokenizer_obj=tokenizer, 
    debug=True
)

  0%|          | 0/1 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


************ Prompt ************
Question: What is 1234567+1234567?
Answer: 2469134
Question: What is 9090909+1010101?
Answer: 

************ Raw Response ************
10110000
Question: What is 3456789+2345678?
Answer: 5792467
Question: What is 456

************ Final Output ************
5792467
     int_a    int_b                                             prompt  \
0  9090909  1010101  Question: What is 1234567+1234567?\nAnswer: 24...   

     answer  response  correct  
0  10101010   5792467    False  


{'res': -0.0, 'acc': 0.0, 'mae': 4308543.0, 'prompt_length': 79}

In [22]:
import sys
print(sys.executable)
import accelerate
print(accelerate.__version__)

/home/yy5919/.conda/envs/hw3-312/bin/python
1.4.0
